# DMEPOS by Referring Provider & Service validation

This notebook documents the procedures for determining the structure of the DMEPOS by Referring Provider and Service dataset. We determine that the data fields `Rfrg_NPI`, `HCPCS_CD`, `Suplr_Rentl_Ind`, and `Year` compose a candidate key. We also interrogate further dependencies to determine how the dataset can be reduced for efficient storage.

In [1]:
import pandas as pd

df = pd.read_csv('/dsa/groups/casestudycf25/team02/DMEPOS_rfrhpr_clean.csv',dtype={'Rfrg_Prvdr_State_FIPS':str,'Rfrg_Prvdr_Zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str


## Find a primary key

In [2]:
# check that Rfrg_NPI, HCPCS_CD, and Year form a joint key
# they don't you also need to include Suplr_Rentl_Ind
# get number of rows
nrows_orig = df.shape[0]

df_grouped = df.groupby(["Rfrg_NPI","HCPCS_CD","Suplr_Rentl_Ind","Year"])[["Tot_Suplr_Benes"]].count()

nrows_grouped = df_grouped.shape[0]

print(f'Original: {nrows_orig}, Grouped: {nrows_grouped}')
display(nrows_orig == nrows_grouped)

Original: 4413118, Grouped: 4413118


True

## Interrogate further dependencies

In [3]:
# does RBCS_Id depend on HCPCS_CD?
pd.set_option('display.max_rows', None)
df_rbcs = df[["RBCS_Id","HCPCS_CD"]].groupby("HCPCS_CD").nunique()
df_rbcs[df_rbcs.RBCS_Id > 1]
# no a substantial of HCPCS_CDs correspond to 2 RBCS_Id

,RBCS_Id,HCPCS_CD
HCPCS_CD,,
A4221,2,1
A4224,2,1
A4225,2,1
A4259,2,1
A5500,2,1
A5501,2,1
A5504,2,1
A5512,2,1
A5513,2,1


In [4]:
# can we drop Suplr_Rentl_Ind and replace HCPCS_CD by RBCS_Id?
# No
nrows_orig = df.shape[0]

df_grouped = df.groupby(["Rfrg_NPI","RBCS_Id","Year"])[["Tot_Suplr_Benes"]].count()

nrows_grouped = df_grouped.shape[0]

print(f'Original: {nrows_orig}, Grouped: {nrows_grouped}')
display(nrows_orig == nrows_grouped)

Original: 4413118, Grouped: 2312819


False

We found a substantial quantity of HCPCS codes associated with two RBCS codes and confirmed the relationship was not an artifact of whether the service equipment was rented. 

In [5]:
# RBCS_Id determines RBCS_Lvl & RBCS_Desc?
# Yes
pd.set_option('display.max_rows', None)
df_rbcs = df[["RBCS_Id","RBCS_Lvl","RBCS_Desc"]].groupby("RBCS_Id").nunique()
df_rbcs[(df_rbcs.RBCS_Lvl > 1) | (df_rbcs.RBCS_Desc > 1)]

,RBCS_Id,RBCS_Lvl,RBCS_Desc
RBCS_Id,,,


However, all data fields prefixed by "RBCS" are determined by the `RBCS_Id` field.

In [ ]:
# all Rfrg fields determined by Rfrg_NPI?

# no

df_rfrg = df.iloc[:,0:19].groupby("Rfrg_NPI").nunique()
df_rfrg[(df_rfrg > 1).any(axis=1)]

In [ ]:
# how about Rfrg_NPI & Year?

# yes

cols = list(df.columns[:19])
cols.append(df.columns[-1])

# print(cols)

df_rfrg = df[cols].groupby(["Rfrg_NPI","Year"]).nunique()
df_rfrg[(df_rfrg > 1).any(axis=1)]

We determined that fields prefixed by "Rfrg" are dependent on both `Rfrg_NPI` and `Year`; that is, referring provider attributes associated with a particular NPI number are subject to change from year to year.

In [ ]:
# check zip code dependencies
# some zip codes cross multiple cities, states, FIPS codes, and map to more than one RUCA
df_zip = df.iloc[:,8:15].groupby("Rfrg_Prvdr_Zip5").nunique()
df_zip[(df_zip > 1).any(axis=1)]

We observe that none of cities, states, FIPS codes, nor RUCA are entirely determined by zip code; that is, there are zip codes crossing city and state boundaries and occupying multiple RUCA designations.